# nano Beatrix — the fourth pass: the sized and the sliced frames

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pass4.ipynb)

Where a frame begins is information; where it ends should not be. In the first three passes the open tag of a frame cost 6.4-6.7 bits at every size, while the close tag cost 0.04 bits at 32 bytes (the records fill the frame and the count settles it) and 2.3-3.8 bits at 128-1,024 bytes, where the size is unannounced. This pass gives the craft its closures by construction, in two forms, and reads them against the plain frames on the same card: both twins, two fresh seeds (3 and 4), stage A at 1,024 context, then the continuation at 4,096.

- **sized**: every open tag carries one byte naming its frame's generic size (a = 32, b = 64, c = 128, d = 256, e = 512, f = 1,024, g = 2,048), as the register tag is MODE + one letter. The craft can count to the close.
- **sliced**: the sized tag, and every frame larger than 32 bytes built as closed 32-byte slices back to back (its open tag, the slices, its END). The closure the craft already reads for free at 32 bytes then sits at every size. A lookup frame's records fill its first slices; a nested lookup frame becomes a run of lookup slices.
- **plain**: the frames of the first three passes, rerun beside them on the same seeds. The stored seed-1 and seed-2 crafts stay on the Hub as a second sample.

The reads: the close tag's bits (near 0 if the closure is free), the open tag's bits, the body bits, the whole value by frame size after the continuation (Castor's stored crafts read 0.52 and 0.33 at 1,024 bytes), the plain text and the canary rows. One difference of capacity is built in: a sliced 128-byte frame holds four records, one a slice, where the plain and sized frames hold seven; every other size holds the recipe's eight records in all three forms. The last run continues Pollux's seed-2 stage-A craft of the third pass on eight-record frames, the one missing cell of the stored plain series.

Twenty-five runs, one at a time, about two and a half hours on the RTX PRO 6000; every finished run ships as it lands. The cells are the third pass's: C1 pins the trainer (0.1.21) and holds the decision block; C2 builds the data, prints each grid's plan, runs 200 steps of the sliced continuation composition for the price line and probes the push path; C3 is the session; C4 renders the results table (its form column names plain, sized or sliced). Put a Hugging Face token with write access to the training repo in this notebook's secrets as HF_TOKEN before C1; tokenless, everything runs and nothing ships.


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.21"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 21), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here (its record and its weights); tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed: 160 MB covers 12,000 steps x 8 x 4,096 four times over
BASE = dict(
    arms='castor-both,pollux-both',   # the twins at the decided bank (table + address, 2026-10-12): the best text on both bodies on two seeds
    seeds='3,4',                      # two fresh seeds for all three forms (the plain stage-A crafts of seeds 1 and 2 already hold their names on the Hub)
    register='caption',               # the register of every pass so far
    frames='dyn', plant=0.75,         # three rows in four framed, one in four a plain document (an all-framed diet made a frame specialist, 2026-10-13)
    frame_records=8, nest=0.3, overlap=0.0, ask='any', tags='mode',   # eight records a lookup frame: the count that forms at 1,024 and carries to 4,096
    log_every=100,                    # the trainer's progress line (step, bits, tok/s, time left) feeds the step bar
)
STAGE_A = dict(BASE, ctx=1024, mb=32, steps=12000, sizes='32,128,256')       # stage A: every copy formed here in 12,000 steps (2026-10-13)
CONT = dict(BASE, ctx=4096, mb=8, steps=4000, sizes='32,128,256,512,1024')    # the continuation: 4,000 steps at 4,096 from the run's stage-A craft
SIZED = dict(sized=1)                 # every open tag names its frame's generic size in one byte
SLICED = dict(sized=1, sliced=32)     # and every frame above 32 bytes is closed 32-byte slices, its own open tag and END around them
GRIDS = dict(
    # 1. stage A in the three forms (12 runs): the copy forms at 1,024 x 32
    stageA_plain=dict(STAGE_A),
    stageA_sized=dict(STAGE_A, **SIZED),
    stageA_sliced=dict(STAGE_A, **SLICED),
    # 2. each stage-A craft continued at 4,096 x 8 with frames to 1,024 bytes, in its own form (12 runs)
    cont_plain=dict(CONT, init='{arm}_s{seed}_L12000_frdyn_cap_mode_sz32-128-256_r8_n30_o0_askany_p75'),
    cont_sized=dict(CONT, **SIZED, init='{arm}_s{seed}_L12000_frdyn_cap_mode_sized_sz32-128-256_r8_n30_o0_askany_p75'),
    cont_sliced=dict(CONT, **SLICED, init='{arm}_s{seed}_L12000_frdyn_cap_mode_sized_sl32_sz32-128-256_r8_n30_o0_askany_p75'),
    # 3. the stored plain series' missing cell: Pollux seed 2's stage-A craft (the third pass) continued on eight-record frames (1 run)
    cont_plain_pollux_s2=dict(CONT, arms='pollux-both', seeds='2', init='{arm}_s{seed}_L12000_frdyn_cap_mode_sz32-128-256_r8_n30_o0_askany_p75'),
)
SMOKE_STEPS = 200                     # C2's short run on this card for the price line; its push is off
PARALLEL = 1                          # runs at once on the card. ONE: a 9M craft at 8 x 4,096 saturates the RTX PRO 6000's compute alone (863k tok/s);
                                      # four at once ran at ~214k each, their sum one run's rate (2026-10-13): spare VRAM is not spare compute. More than one pays
                                      # only when a single run leaves the card idle, and the proof is the step bars' per-run tok/s summing above one run's rate.
print('decision block:', ' + '.join(f'{k} ({len(g)} keys)' for k, g in GRIDS.items()), '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')


In [ ]:
# C2 (P) — PREFLIGHT: the data, each grid's plan, a short run of the hardest composition on this card (the price line), the push probe. No grid training.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner

# the data: the corpus and the held-out head from fineweb-edu (tokenless; a few minutes; the lines carry the time left)
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

# the plans: what each grid expands to, and what this folder already holds
plans = {}
for name, g in GRIDS.items():
    print('== grid', name)
    plans[name] = bed_runner.run(g, out=OUT, dry=True)

# the price line: the pass's hardest composition (the sliced frames to 1,024 bytes at 4,096 x 8, eight records a frame) for SMOKE_STEPS on this
# card (the step bar runs; the record stays local under its own short-run name and is never shipped)
smoke = dict(CONT, **SLICED, arms=BASE['arms'].split(',')[0], seeds='1', steps=SMOKE_STEPS, log_every=20)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any grid'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
todo = sum(p['runs'] - p['done_before'] for p in plans.values())
tokens = sum((p['runs'] - p['done_before']) * int(GRIDS[n]['steps']) * int(GRIDS[n]['ctx']) * int(GRIDS[n]['mb']) for n, p in plans.items())
print(f"this card: {last['tok_s']}k tok/s at 4,096 x 8 (sliced) -> the {todo} runs left about {tokens / (last['tok_s'] * 1e3) / 3600:.1f} h of "
      f"training one at a time, plus about a minute of gauges and the ship a run")

# the push probe (the ship-complete law): a write to the training repo and a read-back BEFORE any training. Tokenless = disarmed.
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_pass4.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'grids': GRIDS}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano frame pass 4: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')


In [ ]:
# C3 (T) — THE SESSION: the grids one after the other, PARALLEL runs at once inside each, resume-first (runner_done.txt in OUT); a STOP file
# in OUT starts no new run, lets the running ones finish and skips the grids after; a grid bar over the runs and a step bar per running child fed
# by the trainer's own progress lines; the job-size and gauge lines print through the bars; every finished run ships to REPO in the background
# as it lands: a Hub fault (a 504) retries with a doubling wait and never stops the grid; a record left unshipped is re-shipped at the next
# start and at the end (runner_shipped.txt; the result's 'unshipped' names any that failed). A restart resumes from the records on disk.
# An interrupt loses only the running runs (the bed has no mid-run checkpoint); rerun the cell to continue.
from geolip.nano_beatrix import bed_runner
for name, g in GRIDS.items():
    res = bed_runner.run(g, out=OUT, push=REPO if (HF_TOKEN and PUSH_OK) else '', parallel=PARALLEL)
    print('grid', name, ':', {k: v for k, v in res.items() if k != 'ran'}, '| ran', len(res['ran']))
    if res['stopped']:
        print('STOP seen: the grids after', name, 'are not started; remove the file and rerun the cell to continue')
        break


In [ ]:
# C4 — THE RESULTS: the index over this session's records; the framed rows table carries the text, the bits outside the frames, the data
# frames' body bits by size, the whole value fetched from a frame by its size, the nested whole and the cost. Run any time; the Hub holds the same records.
from IPython.display import Markdown, display
from geolip.nano_beatrix.bed_table import render
display(Markdown(render(f'{OUT}/bed', pairs=True)))


### The other grids

Add, drop or edit an entry of `GRIDS` in C1, rerun C1 (the install is cached), then C3; finished runs are skipped.

| grid | the entry in `GRIDS` |
|---|---|
| the forms on sixteen-record frames (the continuation's count) | `cont16_sliced=dict(CONT, **SLICED, frame_records=16, init=...)` with the sliced stage-A tag; the same for plain and sized |
| the scrutiny diet in the sliced form (nesting 0.6, a key defined twice in 0.3 of the rows) | `scrutiny_sliced=dict(STAGE_A, **SLICED, frame_records=4, nest=0.6, overlap=0.3, ask='nested,overlap')`, then its continuation |
| the third pass (the scrutiny reads by continuation, the count on the second seeds, Pollux's hub address axis) | `nano_pass3.ipynb` |

A smaller card: `mb=4` at 4,096 (the step count stays; a run takes longer). The session folder is the Colab disk: the Hub copy is the record; a new session rebuilds the corpus in C2, fetches each starting craft from the Hub, and resumes nothing unless `/content/out/runner_done.txt` is restored.
